<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/experiments/first_passage/duffing_oscillator/duffing_mean_escape_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duffing oscillator mean escape time

This notebook is the single public Duffing example. It follows one equation and one parameter set from trajectory simulation through the boundary-adapted PINN and Monte Carlo evaluation:

$$\zeta=0.08,\qquad \epsilon=10^{-5},\qquad R=2.$$

These are the parameter values selected from the trajectory study for the coherent public example. The equation, simulator, PDE residual, and evaluation all use the same values.

The default `quick` model is the same shallow width-64 `tanh` network used by the committor notebook. The `full` preset restores the four-hidden-layer ReQU network from the source Duffing experiment.


## 1. Problem

The stochastic Duffing oscillator is

$$
dX_1=X_2\,dt+\sqrt{2\epsilon}\,dB_1,
\qquad
dX_2=(-X_1-X_1^3-2\zeta X_2)\,dt+\sqrt{2\zeta}\,dB_2.
$$

For the first exit from the disk $\Omega=B_R(0)$, the mean escape time $\tau(x)=\mathbb E_x[T]$ solves

$$\mathcal L\tau=-1\quad\text{in }\Omega,\qquad \tau=0\quad\text{on }\partial\Omega,$$

$$\mathcal L\tau=x_2\tau_{x_1}-(x_1+x_1^3+2\zeta x_2)\tau_{x_2}
+\epsilon\tau_{x_1x_1}+\zeta\tau_{x_2x_2}.$$

No pretrained Duffing PINN result is bundled. Numerical values appear only after this notebook has been run.


In [ ]:
import json
import math
import random
import secrets
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)

## 2. Editable configuration

The quick preset uses a small tanh network and shorter simulation and training runs. The full preset restores the source ReQU architecture and long schedule. Every new run draws fresh training and Monte Carlo seeds and writes to a unique folder with a manifest.

The default run mode is new. Choose resume or evaluate only after pasting that run's full folder path into EXISTING_RUN_DIR. Resume checks saved settings and restores random-generator states from the last checkpoint. Set SAVE_TO_DRIVE = False to save under the runtime's local runs/duffing folder; local files disappear when a hosted runtime is deleted.

In [ ]:
RUN_SCALE = "quick"                 # "quick" or "full"
RUN_MODE = "new"                    # "new", "resume", or "evaluate"
EXISTING_RUN_DIR = ""               # required for resume/evaluate; paste the full run-folder path
RUN_TRAINING = False                # set True only when you intend to train
RUN_SMOKE_CHECK = True
RUN_LBFGS = False
SAVE_TO_DRIVE = True

radius = 2.0
zeta = 0.08
epsilon = 1e-5
power = 2

if RUN_SCALE == "full":
    width, depth = 40, 4
    activation_name = "requ2"
    initialization_gain = 1.5
    collocation_count, batch_size = 2048, 256
    adam_epochs, number_of_decays = 40_000, 400
    learning_rate, final_factor = 1e-2, 5e-2
    lbfgs_iterations = 1_000
    mc_anchors, mc_paths, mc_dt, mc_max_time = 100, 256, 1e-3, 200.0
    report_every, checkpoint_every = 100, 500
elif RUN_SCALE == "quick":
    width, depth = 64, 2
    activation_name = "tanh"
    initialization_gain = None
    collocation_count, batch_size = 512, 128
    adam_epochs, number_of_decays = 40, 20
    learning_rate, final_factor = 1e-2, 5e-2
    lbfgs_iterations = 20
    mc_anchors, mc_paths, mc_dt, mc_max_time = 20, 32, 2e-3, 5.0
    report_every, checkpoint_every = 10, 10
else:
    raise ValueError("RUN_SCALE must be 'quick' or 'full'")

if RUN_MODE not in {"new", "resume", "evaluate"}:
    raise ValueError("RUN_MODE must be 'new', 'resume', or 'evaluate'.")
if RUN_MODE == "evaluate" and RUN_TRAINING:
    raise ValueError("Evaluation mode cannot start training; set RUN_TRAINING = False.")

if drive is not None and SAVE_TO_DRIVE:
    drive.mount("/content/drive", force_remount=False)
    output_root = Path("/content/drive/MyDrive/boundary-adapted-pinns-runs/duffing")
else:
    output_root = Path("runs/duffing")
output_root.mkdir(parents=True, exist_ok=True)

saved_manifest = None
if RUN_MODE == "new":
    SEED = secrets.randbits(32)
    MC_SEED = secrets.randbits(32)
    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    run_dir = output_root / f"{RUN_SCALE}_{run_stamp}_seed{SEED}"
    run_dir.mkdir(parents=False, exist_ok=False)
else:
    if not EXISTING_RUN_DIR.strip():
        raise ValueError("Paste the full path of the intended run into EXISTING_RUN_DIR.")
    run_dir = Path(EXISTING_RUN_DIR).expanduser()
    manifest_path = run_dir / "run_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"{manifest_path} is missing. Resume/evaluate only runs created by this version "
            "of the notebook; older shared-folder outputs have no run manifest."
        )
    with manifest_path.open(encoding="utf-8") as stream:
        saved_manifest = json.load(stream)
    SEED = int(saved_manifest["seed"])
    MC_SEED = int(saved_manifest["mc_seed"])

def seed_all(seed):
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

def atomic_torch_save(payload, path):
    temporary_path = path.with_name(path.name + ".tmp")
    atomic_torch_save(payload, temporary_path)
    temporary_path.replace(path)


def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng_state(state):
    if not state:
        raise ValueError("Checkpoint has no random-generator state; exact resume is unavailable.")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])

seed_all(SEED)

experiment_settings = {
    "run_scale": RUN_SCALE, "seed": SEED, "mc_seed": MC_SEED,
    "radius": radius, "zeta": zeta, "epsilon": epsilon, "power": power,
    "width": width, "depth": depth, "activation_name": activation_name,
    "initialization_gain": initialization_gain,
    "collocation_count": collocation_count, "batch_size": batch_size,
    "adam_epochs": adam_epochs, "number_of_decays": number_of_decays,
    "learning_rate": learning_rate, "final_factor": final_factor,
    "mc_anchors": mc_anchors, "mc_paths": mc_paths, "mc_dt": mc_dt,
    "mc_max_time": mc_max_time,
}

def require_matching_settings(record, expected, saved_path):
    if record.get("settings") != expected:
        raise ValueError(
            f"{saved_path} was created with different settings. Restore the saved configuration "
            "or start a new run; files are never reused automatically."
        )

manifest_path = run_dir / "run_manifest.json"
if RUN_MODE == "new":
    run_manifest = {
        "format_version": 1,
        "run_id": run_dir.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seed": SEED,
        "mc_seed": MC_SEED,
        "settings": experiment_settings,
    }
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(run_manifest, stream, indent=2)
else:
    require_matching_settings({"settings": saved_manifest.get("settings")}, experiment_settings, manifest_path)

print({
    "scale": RUN_SCALE, "mode": RUN_MODE, "run_dir": str(run_dir),
    "training_seed": SEED, "monte_carlo_seed": MC_SEED,
    "zeta": zeta, "epsilon": epsilon,
})

## 3. Simulate a trajectory

Euler–Maruyama advances both components until the path exits the disk or reaches the time cap. This plot checks that the same equation and noise strengths used by the PDE are being simulated.


In [ ]:
def duffing_drift(points):
    x1, x2 = points[..., 0], points[..., 1]
    return torch.stack((x2, -x1 - x1**3 - 2 * zeta * x2), dim=-1)


def simulate_trajectory(
    start=(0.0, 0.0), *, zeta_value=zeta, epsilon_value=epsilon,
    radius_value=radius, dt=1e-3, max_time=100.0, seed=SEED + 1,
):
    rng = np.random.default_rng(seed)
    point = np.asarray(start, dtype=float).copy()
    path = [point.copy()]
    exited = np.linalg.norm(point) >= radius_value
    noise = np.array([math.sqrt(2 * epsilon_value * dt), math.sqrt(2 * zeta_value * dt)])
    for _ in range(int(max_time / dt)):
        if exited:
            break
        drift = np.array([point[1], -point[0] - point[0]**3 - 2 * zeta_value * point[1]])
        point = point + drift * dt + noise * rng.standard_normal(2)
        path.append(point.copy())
        exited = np.linalg.norm(point) >= radius_value
    return np.asarray(path), (len(path) - 1) * dt, exited


trajectory, trajectory_time, trajectory_exited = simulate_trajectory(max_time=5.0 if RUN_SCALE == "quick" else 100.0)
angle = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(6, 6))
plt.plot(trajectory[:, 0], trajectory[:, 1], lw=1)
plt.plot(radius * np.cos(angle), radius * np.sin(angle), "k--")
plt.scatter(*trajectory[0], label="start")
plt.scatter(*trajectory[-1], marker="x", label="last point")
plt.gca().set_aspect("equal")
plt.legend()
plt.title(f"Duffing trajectory: {'exit' if trajectory_exited else 'censored'} at t={trajectory_time:.3g}")
plt.show()


## 4. Boundary-adapted network

The boundary-adapted trial function is

$$\tau_\theta(x)=N_\theta(x)^2\,\frac{R^2-\lVert x\rVert^2}{2R}.$$

Squaring the raw output keeps the interior prediction nonnegative, and the distance factor imposes the zero boundary value exactly. `CoordinateNetwork` and `make_base_network()` below supply the selected default but may be replaced by any `nn.Module` returning one scalar per point. The boundary wrapper does not otherwise constrain its architecture. The training loss is the mean squared strong PDE residual; there is no additional boundary or Monte Carlo term.


In [ ]:
def sample_disk(count, *, device=DEVICE):
    angle = 2 * math.pi * torch.rand(count, device=device)
    radial = radius * torch.sqrt(torch.rand(count, device=device))
    return torch.stack((radial * torch.cos(angle), radial * torch.sin(angle)), dim=1)


def smooth_distance(points):
    return (radius**2 - points.square().sum(dim=1)) / (2 * radius)


class RePU(nn.Module):
    def forward(self, values):
        return torch.relu(values).pow(power)


def make_activation():
    if activation_name == "tanh":
        return nn.Tanh()
    if activation_name == "requ2":
        return RePU()
    raise ValueError(f"Unknown supplied activation: {activation_name}")


class CoordinateNetwork(nn.Module):
    """Supplied default; replace this class or make_base_network() freely."""

    def __init__(self):
        super().__init__()
        layers = [nn.Linear(2, width), make_activation()]
        for _ in range(depth - 1):
            layers.extend((nn.Linear(width, width), make_activation()))
        layers.append(nn.Linear(width, 1))
        self.layers = nn.Sequential(*layers)
        if initialization_gain is not None:
            self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight, gain=initialization_gain)
            nn.init.constant_(module.bias, initialization_gain)

    def forward(self, points):
        return self.layers(points)


def make_base_network():
    """Return any nn.Module with scalar output shape [n] or [n, 1]."""
    return CoordinateNetwork()


class DuffingBoundaryPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = make_base_network()

    def forward(self, points):
        raw = self.network(points)
        if raw.ndim == 1:
            raw = raw.unsqueeze(1)
        if raw.shape != (points.shape[0], 1):
            raise ValueError("The base network must return shape [n] or [n, 1].")
        return raw.square() * smooth_distance(points).clamp_min(0).unsqueeze(1)


def residual(model, points):
    x = points.detach().clone().requires_grad_(True)
    tau = model(x).squeeze(1)
    grad = torch.autograd.grad(tau.sum(), x, create_graph=True)[0]
    tau_xx = torch.autograd.grad(grad[:, 0].sum(), x, create_graph=True)[0][:, 0]
    tau_yy = torch.autograd.grad(grad[:, 1].sum(), x, create_graph=True)[0][:, 1]
    operator = (x[:, 1] * grad[:, 0]
                - (x[:, 0] + x[:, 0]**3 + 2 * zeta * x[:, 1]) * grad[:, 1]
                + epsilon * tau_xx + zeta * tau_yy)
    return operator + 1.0


def pde_loss(model, points):
    return residual(model, points).square().mean()


In [ ]:
def run_model_smoke_check():
    model = DuffingBoundaryPINN().to(DEVICE)
    loss = pde_loss(model, sample_disk(8))
    if not torch.isfinite(loss):
        raise RuntimeError("Duffing smoke check produced a non-finite loss")
    loss.backward()
    print("Duffing model smoke check passed.")


if RUN_SMOKE_CHECK:
    run_model_smoke_check()


## 5. Train, save, and resume

Set RUN_TRAINING = True to train. The run folder saves the training and evaluation points, best and final models, last resumable checkpoint, epoch history, Monte Carlo arrays, and evaluation arrays and figures. To continue an interrupted run, set RUN_MODE = "resume" and paste the exact folder path into EXISTING_RUN_DIR. The latest checkpoint restores optimizer and scheduler state together with Python, NumPy, PyTorch, and CUDA random-generator states. Older checkpoints without those states are refused for exact resumption.

In [ ]:
training_points_path = run_dir / "training_points.pt"
if RUN_MODE in {"resume", "evaluate"}:
    if not training_points_path.is_file():
        raise FileNotFoundError(f"Saved training points are missing: {training_points_path}")
    point_record = torch.load(training_points_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(point_record, experiment_settings, training_points_path)
    collocation_points = point_record["collocation"].to(DEVICE)
    evaluation_points = point_record["evaluation"].to(DEVICE)
    print("Loaded this run's training and evaluation points:", training_points_path)
else:
    if training_points_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing run data: {training_points_path}")
    seed_all(MC_SEED)
    collocation_points = sample_disk(collocation_count)
    evaluation_points = sample_disk(collocation_count)
    atomic_torch_save({"collocation": collocation_points.cpu(), "evaluation": evaluation_points.cpu(), "settings": experiment_settings}, training_points_path)
    print("Saved this run's training and evaluation points:", training_points_path)

np.savez_compressed(run_dir / "training_points.npz",
                    collocation=collocation_points.cpu().numpy(),
                    evaluation=evaluation_points.cpu().numpy())
seed_all(SEED)

last_path = run_dir / "last.pt"
best_path = run_dir / "best.pt"
final_path = run_dir / "final.pt"


def train_duffing():
    model = DuffingBoundaryPINN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    step_size = max(1, adam_epochs // number_of_decays)
    gamma = final_factor**(1 / number_of_decays)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    history, start_epoch, best_loss = [], 0, math.inf
    if RUN_MODE == "resume" and last_path.exists():
        state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        require_matching_settings(state, experiment_settings, last_path)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        history, start_epoch, best_loss = state["history"], state["epoch"], state["best_loss"]
        restore_rng_state(state.get("rng_state"))
        print(f"Resuming after epoch {start_epoch}")
    elif RUN_MODE == "resume" and best_path.exists():
        raise ValueError(f"{best_path} exists without a resumable last.pt checkpoint; refusing an inexact resume.")
    elif RUN_MODE == "new" and (last_path.exists() or best_path.exists() or final_path.exists()):
        raise FileExistsError(f"Refusing to reuse checkpoints in {run_dir}")

    started = time.time()
    for epoch in range(start_epoch + 1, adam_epochs + 1):
        permutation = torch.randperm(collocation_count, device=DEVICE)
        batch_losses = []
        for start in range(0, collocation_count, batch_size):
            batch = collocation_points[permutation[start:start + batch_size]]
            optimizer.zero_grad()
            loss = pde_loss(model, batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach()))
        scheduler.step()
        validation = float(pde_loss(model, evaluation_points).detach())
        row = {"epoch": epoch, "training residual": float(np.mean(batch_losses)), "evaluation residual": validation}
        history.append(row)
        if validation < best_loss:
            best_loss = validation
            atomic_torch_save({"model": model.state_dict(), "epoch": epoch, "loss": validation, "settings": experiment_settings}, best_path)
        if epoch % report_every == 0 or epoch == 1 or epoch == adam_epochs:
            print(f"epoch {epoch:6d}/{adam_epochs} | train {row['training residual']:.3e} | eval {validation:.3e} | {time.time() - started:.1f}s", flush=True)
        if epoch % checkpoint_every == 0 or epoch == adam_epochs:
            atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "history": history, "epoch": epoch,
                "best_loss": best_loss, "settings": experiment_settings,
                "rng_state": capture_rng_state(),
            }, last_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["model"])
    if RUN_LBFGS:
        optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=lbfgs_iterations, tolerance_grad=0, tolerance_change=0, line_search_fn="strong_wolfe")
        calls = 0

        def closure():
            nonlocal calls
            calls += 1
            optimizer_lbfgs.zero_grad()
            loss = pde_loss(model, collocation_points)
            loss.backward()
            if calls % 100 == 0:
                print(f"L-BFGS call {calls}: residual {float(loss.detach()):.3e}", flush=True)
            return loss

        optimizer_lbfgs.step(closure)
    atomic_torch_save({"model": model.state_dict(), "history": history, "scale": RUN_SCALE, "parameters": {"zeta": zeta, "epsilon": epsilon, "radius": radius}, "settings": {**experiment_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}}, final_path)
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(run_dir / "training_history.csv", index=False)
    return model, history_frame


model = history = None
if RUN_TRAINING:
    if RUN_MODE == "evaluate":
        raise ValueError("Set RUN_MODE to 'new' or 'resume' before training.")
    model, history = train_duffing()
    display(history.tail())
elif RUN_MODE == "evaluate":
    if not final_path.is_file():
        raise FileNotFoundError(f"Completed model is missing: {final_path}")
    state = torch.load(final_path, map_location=DEVICE, weights_only=False)
    expected = {**experiment_settings, "lbfgs": RUN_LBFGS, "lbfgs_iterations": lbfgs_iterations}
    require_matching_settings(state, expected, final_path)
    model = DuffingBoundaryPINN().to(DEVICE)
    model.load_state_dict(state["model"])
    history = pd.DataFrame(state["history"])
    print("Loaded completed model from this run:", final_path)
else:
    print("No training started. Set RUN_TRAINING = True to train in this new run.")

## 6. Monte Carlo reference at evaluation anchors

This is a direct mean-exit estimate for the same Duffing SDE; the earlier trajectory study did not include a completed Monte Carlo reference. The cell saves its anchor locations, mean times, censoring fractions, and simulation settings. A capped path contributes the cap, so inspect the censoring column before treating the estimate as a reference.


In [ ]:
@torch.no_grad()
def monte_carlo_exit_time(starts, n_paths, dt, max_time):
    starts = starts.to(DEVICE)
    count = starts.shape[0]
    positions = starts[:, None, :].expand(count, n_paths, 2).clone()
    elapsed = torch.zeros(count, n_paths, device=DEVICE)
    alive = torch.ones(count, n_paths, dtype=torch.bool, device=DEVICE)
    steps = int(max_time / dt)
    interval = max(1, steps // 100)
    noise_scale = torch.tensor([math.sqrt(2 * epsilon * dt), math.sqrt(2 * zeta * dt)], device=DEVICE)
    started = time.time()
    for step in range(steps):
        if not alive.any():
            break
        current = positions[alive]
        positions[alive] = current + duffing_drift(current) * dt + torch.randn_like(current) * noise_scale
        elapsed[alive] += dt
        active_indices = alive.nonzero(as_tuple=False)
        exited = positions[alive].square().sum(dim=1) >= radius**2
        if exited.any():
            ids = active_indices[exited]
            alive[ids[:, 0], ids[:, 1]] = False
        if step % interval == 0 or step + 1 == steps:
            print(f"MC step {step + 1:7d}/{steps} | alive {alive.float().mean():6.2%} | {time.time() - started:6.1f}s", flush=True)
    return elapsed.mean(dim=1), alive.float().mean(dim=1)


mc_path = run_dir / "monte_carlo_reference.pt"
mc_settings = {
    "format_version": 2,
    "seed": SEED, "mc_seed": MC_SEED, "radius": radius, "zeta": zeta, "epsilon": epsilon,
    "anchors": mc_anchors, "paths": mc_paths, "dt": mc_dt, "max_time": mc_max_time,
}
if RUN_MODE in {"resume", "evaluate"}:
    if not mc_path.is_file():
        raise FileNotFoundError(f"Saved Monte Carlo data is missing: {mc_path}")
    mc_record = torch.load(mc_path, map_location=DEVICE, weights_only=False)
    require_matching_settings(mc_record, mc_settings, mc_path)
    mc_x, mc_tau, mc_censored = mc_record["x"].to(DEVICE), mc_record["tau"].to(DEVICE), mc_record["censored"].to(DEVICE)
    print("Loaded Monte Carlo reference:", mc_path)
else:
    if mc_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing Monte Carlo data: {mc_path}")
    seed_all(MC_SEED)
    mc_x = sample_disk(mc_anchors)
    mc_tau, mc_censored = monte_carlo_exit_time(mc_x, mc_paths, mc_dt, mc_max_time)
    atomic_torch_save({"x": mc_x.cpu(), "tau": mc_tau.cpu(), "censored": mc_censored.cpu(), "settings": mc_settings}, mc_path)
    pd.DataFrame({"x1": mc_x[:, 0].cpu().numpy(), "x2": mc_x[:, 1].cpu().numpy(),
                  "mean exit time": mc_tau.cpu().numpy(),
                  "censored fraction": mc_censored.cpu().numpy()}).to_csv(run_dir / "monte_carlo_reference.csv", index=False)
    print("Saved Monte Carlo reference:", mc_path)

display(pd.DataFrame({"mean exit time": mc_tau.cpu().numpy(), "censored fraction": mc_censored.cpu().numpy()}).describe())


## 7. Evaluate the learned solution

The contour plot shows the learned field. At the Monte Carlo anchors, the scatter plot and error table compare like with like. Large censoring fractions indicate that `mc_max_time` or the path count should be increased before drawing a numerical conclusion.


In [ ]:
if model is not None:
    grid_n = 180
    gx, gy = np.meshgrid(np.linspace(-radius, radius, grid_n), np.linspace(-radius, radius, grid_n))
    mask = gx**2 + gy**2 <= radius**2
    points = np.column_stack((gx[mask], gy[mask]))
    with torch.no_grad():
        prediction = model(torch.as_tensor(points, device=DEVICE)).squeeze(1).cpu().numpy()
        anchor_prediction = model(mc_x).squeeze(1).cpu().numpy()
    field = np.full_like(gx, np.nan)
    field[mask] = prediction
    truth = mc_tau.cpu().numpy()
    comparison = pd.DataFrame({
        "PINN": anchor_prediction,
        "Monte Carlo": truth,
        "absolute error": np.abs(anchor_prediction - truth),
        "censored fraction": mc_censored.cpu().numpy(),
    })
    display(comparison)
    comparison.to_csv(run_dir / "anchor_comparison.csv", index=False)
    anchor_rmse = float(np.sqrt(np.mean((anchor_prediction - truth)**2)))
    print("Anchor RMSE:", anchor_rmse)
    np.savez_compressed(run_dir / "evaluation_field.npz", gx=gx, gy=gy, mask=mask.astype(bool),
                        field=field, anchors=mc_x.cpu().numpy(), prediction=anchor_prediction,
                        reference=truth, censored_fraction=mc_censored.cpu().numpy())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    contour = axes[0].contourf(gx, gy, field, levels=50)
    axes[0].set_title("Boundary-adapted PINN")
    axes[0].set_aspect("equal")
    fig.colorbar(contour, ax=axes[0])
    axes[1].scatter(truth, anchor_prediction, c=mc_censored.cpu().numpy(), cmap="magma")
    lo, hi = min(truth.min(), anchor_prediction.min()), max(truth.max(), anchor_prediction.max())
    axes[1].plot([lo, hi], [lo, hi], "k--")
    axes[1].set(xlabel="Monte Carlo mean exit time", ylabel="PINN", title="Anchor comparison; color = censored fraction")
    fig.savefig(run_dir / "evaluation.png", dpi=180, bbox_inches="tight")
    with (run_dir / "evaluation_metrics.json").open("w", encoding="utf-8") as stream:
        json.dump({"anchor_rmse": anchor_rmse}, stream, indent=2)
    plt.show()


## Adapting the method to another PDE

Replace the drift, diffusion coefficients, domain sampler, distance factor, and strong residual together; derive the new generator first so the residual signs and second-order coefficients stay consistent. The trajectory and Monte Carlo functions should use the same SDE and exit rule as the PDE. `make_base_network()` accepts any scalar PyTorch model, while `DuffingBoundaryPINN` keeps the nonnegative exact-boundary trial function separate from that architectural choice.


## References

- J. Qian, L. Chen, and J. Q. Sun, [“A neural network solution of first-passage problems”](https://doi.org/10.1007/s10483-024-3189-8), *Applied Mathematics and Mechanics (English Edition)* 45(11), 2023–2036 (2024).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, [“Boundary-Adapted PINNs for Elliptic Dirichlet Problems”](https://arxiv.org/abs/2607.19167), 2026.
- N. Sukumar and A. Srivastava, [“Exact imposition of boundary conditions with distance functions”](https://doi.org/10.1016/j.cma.2021.114333), *CMAME* 389 (2022).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, [“Physics-informed neural networks”](https://doi.org/10.1016/j.jcp.2018.10.045), *JCP* 378 (2019).
